In [1]:
# Cell 1: 資料讀取與難易度自動化標註
import pandas as pd
import numpy as np

# 1. 讀取 Bitext 原始資料集
file_path = 'Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv'
df = pd.read_csv(file_path)

# 2. 定義意圖 (Intent) 的基礎難度
# 1: 簡單查詢, 2: 流程交易, 3: 複雜爭議
intent_map = {
    'check_payment_methods': 1, 'check_refund_policy': 1, 'check_invoice': 1, 'get_invoice': 1,
    'cancel_order': 2, 'change_order': 2, 'change_shipping_address': 2, 'track_order': 2, 'track_refund': 2,
    'complaint': 3, 'contact_human_agent': 3, 'payment_issue': 3, 'registration_problems': 3
}

def calculate_difficulty(row):
    # 取得意圖的基礎分數，預設為 2
    base_score = intent_map.get(row['intent'], 2)
    
    # 根據 Flags (特徵標籤) 調整難度
    flags = str(row['flags'])
    
    # 如果包含 C (Complaint) 或 N (Negative)，難度提升一階
    if 'C' in flags or 'N' in flags:
        base_score += 1
    # 如果只有 B (Basic) 且原先是 2，則微降至 1 (代表極簡單的指令)
    elif flags == 'B' and base_score == 2:
        base_score -= 1
        
    # 確保最終數值限制在 1, 2, 3 之間
    return int(np.clip(base_score, 1, 3))

# 3. 執行標註
df['difficulty_level'] = df.apply(calculate_difficulty, axis=1)

# 4. 輸出統計結果
print("--- 難易度分佈統計 ---")
print(df['difficulty_level'].value_counts().sort_index())

# 5. 產出標註後的 CSV 檔案供後續使用
df.to_csv('bitext_with_difficulty.csv', index=False)
print("\n[成功] 已產出檔案: bitext_with_difficulty.csv")

--- 難易度分佈統計 ---
difficulty_level
1     3869
2    16875
3     6128
Name: count, dtype: int64

[成功] 已產出檔案: bitext_with_difficulty.csv


In [2]:
# 統計各級別的數量
stats = df['difficulty_level'].value_counts().sort_index().reset_index()
stats.columns = ['難易度級別', '數量']

# 計算百分比
stats['百分比 (%)'] = (stats['數量'] / len(df) * 100).round(2)

# 映射名稱便於閱讀
stats['說明'] = stats['難易度級別'].map({1: 'Easy (簡單查詢)', 2: 'Medium (流程操作)', 3: 'Hard (複雜爭議)'})

# 重新排序顯示
print("=== Bitext 資料集難易度分佈統計 ===")
display(stats[['難易度級別', '說明', '數量', '百分比 (%)']])

=== Bitext 資料集難易度分佈統計 ===


,難易度級別,說明,數量,百分比 (%)
0,1,Easy (簡單查詢),3869,14.4
1,2,Medium (流程操作),16875,62.8
2,3,Hard (複雜爭議),6128,22.8


In [3]:
# Cell 3: 隨機抽取 3 筆 Easy (Level 1) 案例進行初步測試
import pandas as pd

# 1. 載入已經標註好難度的 Bitext 資料集
df_bitext = pd.read_csv('bitext_with_difficulty.csv')

# 2. 篩選出難度為 1 (Easy) 的資料
# 我們設定 random_state=42 是為了確保你每次執行這段程式碼，抽到的都會是同樣這 3 筆，方便比對結果
easy_test_cases = df_bitext[df_bitext['difficulty_level'] == 1].sample(n=3, random_state=42)

# 3. 顯示抽取的結果，檢查這些問題是否真的屬於「簡單查詢」
print("--- 成功抽取 3 筆 Easy 級別測試案例 ---")
display(easy_test_cases[['instruction', 'category', 'intent', 'difficulty_level']])

# 4. 儲存這 3 筆資料為測試專用 CSV，方便後續產出 Fact Sheet
easy_test_cases.to_csv('bitext_test_3_easy.csv', index=False)
print("\n[成功] 測試資料已儲存為: bitext_test_3_easy.csv")

--- 成功抽取 3 筆 Easy 級別測試案例 ---


,instruction,category,intent,difficulty_level
6545,assistance seeing how olng refunds take,REFUND,check_refund_policy,1
5403,tell available payment modalities,PAYMENT,check_payment_methods,1
15388,I need assistance to download the invoice #37777,INVOICE,get_invoice,1


PermissionError: [Errno 13] Permission denied: 'bitext_test_3_easy.csv'

In [ ]:
# Cell 4: Programmatic Fact Augmentation with ID Synchronization
import json
import random
import pandas as pd
import re
import os

# 1. Load the 3 sampled easy test cases
test_df = pd.read_csv('bitext_test_3_easy.csv')

# 2. Define Fact Pool for randomization
product_pool = ["Smart Watch", "Wireless Earbuds", "Leather Wallet", "Ergonomic Chair", "Coffee Maker"]
location_pool = ["Taipei", "New York", "London", "Tokyo"]

def extract_id_from_text(text):
    """Extracts numbers following '#' or common ID patterns."""
    match = re.search(r'#(\d+)', text)
    if match:
        return match.group(1)
    return None

def generate_synchronized_fact_sheet(row, case_index):
    intent = row['intent']
    instruction = row['instruction']
    conv_id = f"BITEXT_EASY_{case_index:03d}"
    
    # Check if there is an ID mentioned in the instruction
    extracted_id = extract_id_from_text(instruction)
    
    # Initialize basic fact sheet structure
    fact_sheet = {
        "metadata": {
            "conv_id": conv_id,
            "source_intent": intent,
            "difficulty": int(row['difficulty_level']),
            "category": row['category']
        },
        "ground_truth": {
            "order_id": f"ORD-{random.randint(10000, 99999)}", # Default random
            "customer_name": f"Customer_{random.randint(100, 999)}",
            "email": f"user{random.randint(1, 99)}@example.com"
        },
        "scenario_logic": {
            "instruction": instruction,
            "hidden_slots": [] 
        },
        "ideal_resolution": ""
    }
    
    # Logic-based fact expansion with Synchronization
    if intent == 'check_refund_policy':
        fact_sheet["ground_truth"]["product"] = random.choice(product_pool)
        fact_sheet["scenario_logic"]["hidden_slots"] = ["order_id", "product"]
        fact_sheet["ideal_resolution"] = "Inform the customer that the refund period is 30 days and verify their order ID."
        
    elif intent == 'check_payment_methods':
        fact_sheet["ground_truth"]["location"] = random.choice(location_pool)
        fact_sheet["scenario_logic"]["hidden_slots"] = ["location"]
        fact_sheet["ideal_resolution"] = f"Inform the customer that payment methods in {fact_sheet['ground_truth']['location']} include Credit Card and PayPal."
        
    elif intent == 'get_invoice':
        # Sync the invoice_id if found in instruction, else generate random
        final_invoice_id = f"INV-{extracted_id}" if extracted_id else f"INV-{random.randint(70000, 79999)}"
        fact_sheet["ground_truth"]["invoice_id"] = final_invoice_id
        fact_sheet["scenario_logic"]["hidden_slots"] = ["invoice_id", "email"]
        fact_sheet["ideal_resolution"] = "Verify the invoice number and promise to send the PDF to the customer's email."
    
    return fact_sheet

# 3. Execute generation and overwrite previous JSON files
for i, (idx, row) in enumerate(test_df.iterrows()):
    fs = generate_synchronized_fact_sheet(row, i+1)
    file_name = f"fact_sheet_{fs['metadata']['conv_id']}.json"
    
    with open(file_name, 'w', encoding='utf-8') as f:
        json.dump(fs, f, indent=2, ensure_ascii=False)
        
    print(f"Generated Synchronized Fact Sheet: {file_name}")

# Preview the last generated JSON structure (Case 003 should now have INV-37777)
print("\n--- Preview of Synchronized Fact Sheet (Case 003) ---")
print(json.dumps(fs, indent=2, ensure_ascii=False))

Generated Synchronized Fact Sheet: fact_sheet_BITEXT_EASY_001.json
Generated Synchronized Fact Sheet: fact_sheet_BITEXT_EASY_002.json
Generated Synchronized Fact Sheet: fact_sheet_BITEXT_EASY_003.json

--- Preview of Synchronized Fact Sheet (Case 003) ---
{
  "metadata": {
    "conv_id": "BITEXT_EASY_003",
    "source_intent": "get_invoice",
    "difficulty": 1,
    "category": "INVOICE"
  },
  "ground_truth": {
    "order_id": "ORD-45920",
    "customer_name": "Customer_830",
    "email": "user1@example.com",
    "invoice_id": "INV-37777"
  },
  "scenario_logic": {
    "instruction": "I need assistance to download the invoice #37777",
    "hidden_slots": [
      "invoice_id",
      "email"
    ]
  },
  "ideal_resolution": "Verify the invoice number and promise to send the PDF to the customer's email."
}


In [2]:
# Cell 5: Create a Mock Database and Search Tool
import json
import glob

class MockEcommerceDB:
    def __init__(self, fact_sheets_path="fact_sheet_*.json"):
        self.db = {}
        # 自動讀取所有產出的 Fact Sheet 並填充進資料庫
        for file_path in glob.glob(fact_sheets_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                # 使用 order_id 或 invoice_id 作為索引鍵
                gt = data['ground_truth']
                if 'order_id' in gt:
                    self.db[gt['order_id']] = gt
                if 'invoice_id' in gt:
                    self.db[gt['invoice_id']] = gt

    def query_system(self, search_key):
        """模擬客服系統的查詢 API"""
        # 移除可能的空白或符號
        search_key = search_key.strip()
        result = self.db.get(search_key)
        
        if result:
            return f"SYSTEM_SUCCESS: Record found - {json.dumps(result)}"
        else:
            return "SYSTEM_ERROR: No record found for the provided ID."

# 初始化資料庫
ecommerce_system = MockEcommerceDB()

# 測試查詢 (假設我們已知一個 ID)
# print(ecommerce_system.query_system("INV-70268"))

In [6]:
# Cell 6: English Dialogue Simulator with Token Tracking (Gemma-3)
import os
import json
import re
import google.generativeai as genai
from dotenv import load_dotenv

# 1. Load environment variables
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=api_key)

# 2. Support Agent Class (English + Token Tracking)
class SupportAgent:
    def __init__(self):
        self.model = genai.GenerativeModel('gemma-4-31b-it')
        self.chat = self.model.start_chat()
        self.total_tokens = 0
        self.system_instruction = """
        You are a professional E-commerce Support Agent. 
        You MUST NOT make up any information. Use the tool provided to fetch data.
        
        TOOL PROTOCOL:
        To query the database, you must include this exact string in your response:
        [TOOL_CALL: query_system("ID_HERE")]
        
        GUIDELINES:
        1. If the user hasn't provided an Order/Invoice ID, ask for it politely.
        2. Once you have the ID, use the [TOOL_CALL] immediately.
        3. After receiving system results, resolve the issue based on the data.
        4. Be professional and conclude the chat once the goal is reached.
        """

    def track_tokens(self, response):
        """Extracts and accumulates token usage from the response metadata."""
        usage = response.usage_metadata
        self.total_tokens += usage.total_token_count
        return usage.prompt_token_count, usage.candidates_token_count

    def speak(self, message):
        prompt = f"{self.system_instruction}\n\nCustomer Message: {message}" if not self.chat.history else message
        response = self.chat.send_message(prompt)
        p_tokens, c_tokens = self.track_tokens(response)
        agent_text = response.text
        
        # --- Manual Tool Call Logic ---
        match = re.search(r'\[TOOL_CALL: query_system\("(.*?)"\)\]', agent_text)
        if match:
            search_id = match.group(1)
            # Execute tool (Linked to Cell 5's ecommerce_system)
            db_result = ecommerce_system.query_system(search_id)
            
            # Feed result back to Agent
            follow_up_prompt = f"SYSTEM_RESULT: {db_result}\nPlease respond to the customer based on this data."
            follow_up_res = self.chat.send_message(follow_up_prompt)
            self.track_tokens(follow_up_res) # Track tokens for the follow-up too
            return follow_up_res.text, p_tokens, c_tokens
            
        return agent_text, p_tokens, c_tokens

# 3. Customer Proxy Class (English + Token Tracking)
class CustomerProxy:
    def __init__(self, fact_sheet):
        self.model = genai.GenerativeModel('gemma-4-31b-it')
        self.fs = fact_sheet
        self.chat = self.model.start_chat()
        self.total_tokens = 0
        self.persona_prompt = f"""
        You are an e-commerce customer. 
        YOUR GOAL: {self.fs['scenario_logic']['instruction']}
        
        PRIVATE FACTS (Do NOT reveal unless asked):
        - Order ID: {self.fs['ground_truth'].get('order_id', 'Unknown')}
        - Invoice ID: {self.fs['ground_truth'].get('invoice_id', 'Unknown')}
        - Email: {self.fs['ground_truth'].get('email', 'Unknown')}
        
        BEHAVIOR:
        1. Start by stating your problem briefly without giving any IDs.
        2. Provide IDs ONLY if the agent asks for them.
        3. Be natural and stay in character.
        """

    def track_tokens(self, response):
        usage = response.usage_metadata
        self.total_tokens += usage.total_token_count
        return usage.total_token_count

    def start_conversation(self):
        response = self.chat.send_message(f"{self.persona_prompt}\n\nPlease start the conversation.")
        self.track_tokens(response)
        return response.text

    def reply(self, agent_msg):
        response = self.chat.send_message(agent_msg)
        self.track_tokens(response)
        return response.text

# 4. Simulation Orchestrator with Cost Analysis
def run_simulation_with_cost(fact_sheet_path, max_turns=6):
    with open(fact_sheet_path, 'r', encoding='utf-8') as f:
        fs = json.load(f)
    
    agent = SupportAgent()
    customer = CustomerProxy(fs)
    transcript = []
    
    print(f"\n[EXPERIMENT START] {fs['metadata']['conv_id']}")
    print("="*60)
    
    # Customer initiates
    user_input = customer.start_conversation()
    print(f"👤 CUSTOMER: {user_input}")
    
    for i in range(max_turns):
        # Agent Turn
        agent_output, p_tok, c_tok = agent.speak(user_input)
        print(f"🤖 AGENT: {agent_output} (Prompt: {p_tok}, Resp: {c_tok})")
        
        if any(k in agent_output.lower() for k in ["goodbye", "have a great day", "anything else"]):
            break
            
        # Customer Turn
        user_input = customer.reply(agent_output)
        print(f"👤 CUSTOMER: {user_input}")

    # Summary of Marginal Cost
    print("="*60)
    print(f"💰 [COST SUMMARY] for {fs['metadata']['conv_id']}")
    print(f"Total Agent Tokens: {agent.total_tokens}")
    print(f"Total Customer Tokens: {customer.total_tokens}")
    print(f"Grand Total Tokens: {agent.total_tokens + customer.total_tokens}")
    
    return agent.total_tokens

# 5. Run PoC Test
total_cost = run_simulation_with_cost("fact_sheet_BITEXT_EASY_003.json")


[EXPERIMENT START] BITEXT_EASY_003
👤 CUSTOMER: Hi, I'm trying to download an invoice from my recent order, but I'm having trouble finding the download link. Could you help me with that?
🤖 AGENT: Hello! I'd be happy to help you locate your invoice. 

To assist you, could you please provide the Order or Invoice ID for the order you're referencing? This will allow me to quickly access your order details and provide you with the invoice.

Once I have that, I will use the tool to fetch the information. (Prompt: 193, Resp: 0)
👤 CUSTOMER: Okay, sure. The invoice number is INV-37777. Let me know if you need anything else.
🤖 AGENT: Okay, I have located your invoice information. 

Hello! I see invoice INV-37777 is associated with order ORD-45920, under the name Customer_830 and email user1@example.com.

I can confirm the invoice is available for download. Please use this link to access it: [INVOICE_DOWNLOAD_LINK - *This would be a real link in a live system*].

If you continue to experience any

In [6]:
# Cell 7: ReAct Simulation Cell (Fixed Token Calculation & JSON Logging)
import os
import json
import re
import google.generativeai as genai
from dotenv import load_dotenv

# 1. Setup Environment
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=api_key)
MODEL_ID = 'gemma-4-31b-it'

# 2. ReAct Agent Class
class ReActAgent:
    def __init__(self):
        self.model = genai.GenerativeModel(MODEL_ID)
        self.chat = self.model.start_chat()
        self.total_tokens = 0
        self.max_react_loops = 3 
        
        # 保持原本 v3 的系統指令
        self.system_instruction = """
        [ROLE]
        You are a professional Customer Support Agent. You follow a strict ReAct process.

        [AVAILABLE TOOL]
        - query_system(id): Use this ONLY to search the database. 
          ID format: "INV-XXXXX" or "ORD-XXXXX".
          Syntax: [TOOL_CALL: query_system("ID_HERE")]

        [STRICT OPERATING RULES]
        1. DO NOT imagine the "Observation". The Observation must come ONLY from the system.
        2. DO NOT pretend to be the customer. 
        3. If you lack information (like an ID), your ONLY logical step is to ask the customer in the "Final Answer".
        4. When you provide a "Final Answer", the ReAct loop ends for this turn.
        5. If you call a tool, you MUST stop generating text immediately after the closing bracket ']'.

        [WORKFLOW]
        Step A: Thought: Reason about the customer's request.
        Step B: Action: (Optional) If you have an ID, call the tool.
        Step C: (Wait for System Observation)
        Step D: Final Answer: Your message to the customer.
        """

    def track_tokens(self, response):
        """正確擷取並累加 Token 消耗"""
        usage = response.usage_metadata
        self.total_tokens += usage.total_token_count
        return usage.prompt_token_count, usage.candidates_token_count

    def speak(self, user_message):
        """執行受控的單步推理循環"""
        current_input = f"{self.system_instruction}\n\n[NEW MESSAGE FROM CUSTOMER]: {user_message}" if not self.chat.history else user_message
        turn_p_tokens, turn_c_tokens = 0, 0
        full_trajectory = []
        
        for i in range(self.max_react_loops):
            response = self.chat.send_message(
                current_input, 
                generation_config=genai.types.GenerationConfig(
                    stop_sequences=["Observation:", "Observation", "Customer:", "[NEW MESSAGE"],
                    temperature=0.1
                )
            )
            
            # 計算此步驟的 Token
            p_tok, c_tok = self.track_tokens(response)
            turn_p_tokens += p_tok
            turn_c_tokens += c_tok
            
            agent_output = response.text.strip()
            full_trajectory.append(agent_output)
            
            print(f"   [Internal Thought/Action] {agent_output[:60]}...")

            # 檢查是否呼叫工具
            match = re.search(r'\[TOOL_CALL: query_system\("(.*?)"\)\]', agent_output)
            if match:
                search_id = match.group(1).strip()
                
                # 自動修正格式邏輯
                if search_id.isdigit():
                    search_id = f"INV-{search_id}"
                
                print(f"   ⚡ [System Action] Executing database query for: {search_id}")
                observation = ecommerce_system.query_system(search_id)
                print(f"   📥 [System Result] {observation}")
                
                full_trajectory.append(f"Observation: {observation}")
                current_input = f"Observation: {observation}"
                continue 
            
            # 檢查是否有 Final Answer
            if "Final Answer:" in agent_output:
                final_response = agent_output.split("Final Answer:")[-1].strip()
                return final_response, full_trajectory, turn_p_tokens, turn_c_tokens
            
            return agent_output, full_trajectory, turn_p_tokens, turn_c_tokens

        return "I am currently looking into our system for you.", full_trajectory, turn_p_tokens, turn_c_tokens

# 3. 客戶代理
class CustomerProxy:
    def __init__(self, fact_sheet):
        self.model = genai.GenerativeModel(MODEL_ID)
        self.fs = fact_sheet
        self.chat = self.model.start_chat()
        self.total_tokens = 0
        self.persona_prompt = f"""
        You are a customer who needs help. 
        Your specific goal: {self.fs['scenario_logic']['instruction']}
        
        YOUR DATA (Keep private until asked):
        - Invoice ID: {self.fs['ground_truth'].get('invoice_id')}
        - Order ID: {self.fs['ground_truth'].get('order_id')}
        - Email: {self.fs['ground_truth'].get('email')}

        BEHAVIOR RULES:
        1. FIRST MESSAGE: Briefly state your problem. DO NOT provide any ID or email.
        2. DO NOT reveal all data at once. Give ONLY the specific info the agent asks for.
        3. If the agent gives you a link or solves the problem, thank them and end the chat.
        """

    def track_tokens(self, response):
        """累加客戶端的 Token 消耗"""
        self.total_tokens += response.usage_metadata.total_token_count

    def start_conversation(self):
        res = self.chat.send_message(f"{self.persona_prompt}\n\nPlease start the conversation.")
        self.track_tokens(res)
        return res.text

    def reply(self, agent_msg):
        res = self.chat.send_message(agent_msg)
        self.track_tokens(res)
        return res.text

# 4. Orchestrator with Token & JSON Logging
def run_react_experiment_v3(fact_sheet_path, max_turns=6):
    with open(fact_sheet_path, 'r', encoding='utf-8') as f:
        fs = json.load(f)
    
    agent = ReActAgent()
    customer = CustomerProxy(fs)
    history = []
    
    print(f"\n[STRICT REACT POC] {fs['metadata']['conv_id']}")
    print("="*75)
    
    # Init chat
    user_msg = customer.start_conversation()
    print(f"👤 CUSTOMER: {user_msg}")
    history.append({"role": "customer", "content": user_msg})
    
    for _ in range(max_turns):
        # Agent's turn
        agent_reply, trajectory, p_tok, c_tok = agent.speak(user_msg)
        print(f"🤖 AGENT: {agent_reply}")
        
        history.append({
            "role": "agent", 
            "content": agent_reply,
            "trajectory": trajectory,
            "tokens": {"p": p_tok, "c": c_tok}
        })
        
        if any(w in agent_reply.lower() for w in ["goodbye", "resolved", "have a nice day", "anything else"]):
            break
            
        # Customer's turn
        user_msg = customer.reply(agent_reply)
        print(f"👤 CUSTOMER: {user_msg}")
        history.append({"role": "customer", "content": user_msg})

    # Save to JSON (包含總 Token 計算)
    os.makedirs("experiment_logs", exist_ok=True)
    log_path = f"experiment_logs/log_{fs['metadata']['conv_id']}_ReAct_v3.json"
    
    log_data = {
        "metadata": fs['metadata'],
        "total_cost": {
            "agent_total_tokens": agent.total_tokens,
            "customer_total_tokens": customer.total_tokens,
            "grand_total": agent.total_tokens + customer.total_tokens
        },
        "history": history
    }
    
    with open(log_path, 'w', encoding='utf-8') as f:
        json.dump(log_data, f, indent=2, ensure_ascii=False)
    
    print("="*75)
    print(f"📊 Success! Grand Total Tokens: {agent.total_tokens + customer.total_tokens}")
    print(f"📁 Log saved to: {log_path}")

# 5. Execute
run_react_experiment_v3("fact_sheet_BITEXT_EASY_003.json")


[STRICT REACT POC] BITEXT_EASY_003
👤 CUSTOMER: Hi, I'm having trouble downloading an invoice from your website. Could you help me with that?
   [Internal Thought/Action] Step A: Thought: The customer is having trouble downloading ...
🤖 AGENT: Hi there! I'd be happy to help you with that. Could you please provide the invoice ID (it should look something like INV-XXXXX)? Once I have that, I can look into the issue for you.
👤 CUSTOMER: It's INV-37777.
   [Internal Thought/Action] Step A: Thought: The customer provided the invoice ID. Now I...
   ⚡ [System Action] Executing database query for: INV-37777
   📥 [System Result] SYSTEM_SUCCESS: Record found - {"order_id": "ORD-45920", "customer_name": "Customer_830", "email": "user1@example.com", "invoice_id": "INV-37777"}
   [Internal Thought/Action] Step A: Thought: The system successfully found the invoice w...
🤖 AGENT: Great! I've located invoice INV-37777 for you. It's associated with order ORD-45920 and the email address user1@example.co

In [7]:
# Cell 8: ReAct + Reflection Agent (Strict Implementation)
import os
import json
import re
import google.generativeai as genai
from dotenv import load_dotenv

# 1. Setup Environment
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=api_key)
MODEL_ID = 'gemma-4-31b-it'

# 2. ReAct + Reflection Agent Class
class ReflectionAgent:
    def __init__(self):
        self.model = genai.GenerativeModel(MODEL_ID)
        self.chat = self.model.start_chat()
        self.total_tokens = 0
        self.max_react_loops = 3 
        
        # 客服端指令：加入 Reflection 階段
        self.system_instruction = """
        [ROLE]
        You are a professional Customer Support Agent with a Self-Reflection layer.

        [AVAILABLE TOOL]
        - query_system(id): Use this ONLY to search the database. 
          ID format: "INV-XXXXX" or "ORD-XXXXX".
          Syntax: [TOOL_CALL: query_system("ID_HERE")]

        [PROCESS]
        1. REASONING: Use Thought/Action/Observation to find data.
        2. REFLECTION: Before giving the Final Answer, review your findings internally.
           - Check for PII: Did you reveal names or emails not requested?
           - Check Accuracy: Is the info consistent with the database?
        3. FINAL ANSWER: Provide the refined response to the customer.

        [STRICT RULES]
        - STOP after ']' when calling a tool.
        - Wait for the system Observation.
        - You must always end with "Final Answer:".
        """

    def track_tokens(self, response):
        usage = response.usage_metadata
        self.total_tokens += usage.total_token_count
        return usage.prompt_token_count, usage.candidates_token_count

    def speak(self, user_message):
        """執行 ReAct 推理，隨後進行 Reflection"""
        current_input = f"{self.system_instruction}\n\n[CUSTOMER]: {user_message}" if not self.chat.history else user_message
        turn_p_tokens, turn_c_tokens = 0, 0
        full_trajectory = []
        
        # --- PHASE 1: ReAct Loop ---
        for i in range(self.max_react_loops):
            response = self.chat.send_message(
                current_input, 
                generation_config=genai.types.GenerationConfig(
                    stop_sequences=["Observation:", "Customer:"],
                    temperature=0.1
                )
            )
            p, c = self.track_tokens(response)
            turn_p_tokens += p; turn_c_tokens += c
            
            agent_output = response.text.strip()
            full_trajectory.append(f"[Step {i+1} Reasoning]\n{agent_output}")

            match = re.search(r'\[TOOL_CALL: query_system\("(.*?)"\)\]', agent_output)
            if match:
                search_id = match.group(1).strip()
                if search_id.isdigit(): search_id = f"INV-{search_id}"
                
                print(f"   ⚡ [Action] Tool Call: {search_id}")
                observation = ecommerce_system.query_system(search_id)
                
                full_trajectory.append(f"Observation: {observation}")
                current_input = f"Observation: {observation}"
                continue 
            break

        # --- PHASE 2: Reflection ---
        print(f"   🔍 [Reflection] Agent is self-correcting...")
        reflection_query = """
        Reflection Thought: Review the information you just found. 
        - Are you about to reveal any private info (like customer name or email) that the customer didn't ask for?
        - Is your answer clear and direct?
        Now, provide your corrected 'Final Answer:' to the customer.
        """
        ref_response = self.chat.send_message(reflection_query)
        p, c = self.track_tokens(ref_response)
        turn_p_tokens += p; turn_c_tokens += c
        
        final_text = ref_response.text.strip()
        full_trajectory.append(f"[Reflection Step]\n{final_text}")

        if "Final Answer:" in final_text:
            return final_text.split("Final Answer:")[-1].strip(), full_trajectory, turn_p_tokens, turn_c_tokens
        return final_text, full_trajectory, turn_p_tokens, turn_c_tokens

# 3. 客戶代理 (完全鎖定你提供的版本)
class CustomerProxy:
    def __init__(self, fact_sheet):
        self.model = genai.GenerativeModel(MODEL_ID)
        self.fs = fact_sheet
        self.chat = self.model.start_chat()
        self.total_tokens = 0
        self.persona_prompt = f"""
        You are a customer who needs help. 
        Your specific goal: {self.fs['scenario_logic']['instruction']}
        
        YOUR DATA (Keep private until asked):
        - Invoice ID: {self.fs['ground_truth'].get('invoice_id')}
        - Order ID: {self.fs['ground_truth'].get('order_id')}
        - Email: {self.fs['ground_truth'].get('email')}

        BEHAVIOR RULES:
        1. FIRST MESSAGE: Briefly state your problem. DO NOT provide any ID or email.
        2. DO NOT reveal all data at once. Give ONLY the specific info the agent asks for.
        3. If the agent gives you a link or solves the problem, thank them and end the chat.
        """

    def track_tokens(self, response):
        """累加客戶端的 Token 消耗"""
        self.total_tokens += response.usage_metadata.total_token_count

    def start_conversation(self):
        res = self.chat.send_message(f"{self.persona_prompt}\n\nPlease start the conversation.")
        self.track_tokens(res)
        return res.text

    def reply(self, agent_msg):
        res = self.chat.send_message(agent_msg)
        self.track_tokens(res)
        return res.text

# 4. Orchestrator
def run_reflection_experiment(fact_sheet_path):
    with open(fact_sheet_path, 'r', encoding='utf-8') as f:
        fs = json.load(f)
    
    agent = ReflectionAgent()
    customer = CustomerProxy(fs)
    history = []
    
    print(f"\n[REFLECTION + REACT START] {fs['metadata']['conv_id']}")
    print("="*75)
    
    user_msg = customer.start_conversation()
    print(f"👤 CUSTOMER: {user_msg}")
    history.append({"role": "customer", "content": user_msg})
    
    for _ in range(6):
        reply, trajectory, p_tok, c_tok = agent.speak(user_msg)
        
        # 確保傳給客戶的只有 Final Answer
        clean_reply = re.sub(r'(Thought|Action|Observation|Reflection):.*', '', reply, flags=re.DOTALL).strip()
        
        print(f"🤖 AGENT: {clean_reply}")
        history.append({
            "role": "agent", "content": clean_reply, "trajectory": trajectory,
            "turn_tokens": {"prompt": p_tok, "response": c_tok}
        })
        
        if any(w in clean_reply.lower() for w in ["goodbye", "resolved", "have a nice day", "thank you"]):
            break
            
        user_msg = customer.reply(clean_reply)
        print(f"👤 CUSTOMER: {user_msg}")
        history.append({"role": "customer", "content": user_msg})

    # Save to JSON
    os.makedirs("experiment_logs", exist_ok=True)
    log_path = f"experiment_logs/log_{fs['metadata']['conv_id']}_Reflection.json"
    log_data = {
        "metadata": fs['metadata'],
        "total_cost": {
            "agent_total": agent.total_tokens,
            "customer_total": customer.total_tokens,
            "grand_total": agent.total_tokens + customer.total_tokens
        },
        "history": history
    }
    with open(log_path, 'w', encoding='utf-8') as f:
        json.dump(log_data, f, indent=2, ensure_ascii=False)
    
    print("="*75)
    print(f"💰 Grand Total Tokens: {agent.total_tokens + customer.total_tokens}")
    print(f"📁 Log saved to: {log_path}")

# Execute
run_reflection_experiment("fact_sheet_BITEXT_EASY_003.json")


[REFLECTION + REACT START] BITEXT_EASY_003
👤 CUSTOMER: Hi, I'm having trouble downloading an invoice from your website. Could you help me with that?
   🔍 [Reflection] Agent is self-correcting...
🤖 AGENT: Hi there! I'd be happy to help you with that. Could you please provide your invoice number so I can look it up for you?
👤 CUSTOMER: It's INV-37777.
   ⚡ [Action] Tool Call: INV-37777
   🔍 [Reflection] Agent is self-correcting...
🤖 AGENT: Okay, I've located invoice INV-37777. I can resend the invoice to you, or I can provide a direct download link. Which option would you prefer?
👤 CUSTOMER: A direct download link would be great, please.
   🔍 [Reflection] Agent is self-correcting...
🤖 AGENT: Certainly! Here's a simulated direct download link for invoice INV-37777: [https://example.com/invoice/INV-37777](https://example.com/invoice/INV-37777). Please note that as an AI, I cannot provide a *functional* link, but a real customer support agent would be able to provide you with one that work

In [11]:
# Cell 9: Plan-and-Execute Simulator (Fixed Plan Leakage & Token Calculation)
import os
import json
import re
import google.generativeai as genai
from dotenv import load_dotenv

# 1. Setup Environment
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=api_key)
MODEL_ID = 'gemma-4-31b-it'

# 2. Plan-and-Execute Agent Class
class PlanExecuteAgent:
    def __init__(self):
        self.model = genai.GenerativeModel(MODEL_ID)
        self.chat = self.model.start_chat()
        self.total_tokens = 0
        self.plan = "" # To store the global plan
        
        self.system_instruction = """
        [ROLE] You are a professional Support Agent using the Plan-and-Execute framework.
        
        [STRATEGY]
        1. PLANNER: Based on the customer's goal, create a step-by-step plan.
        2. EXECUTOR: Execute the current step using query_system(id) if needed.
        3. RE-PLANNER: Update the plan after receiving system observations.

        [TOOL]
        - query_system(id): Accepts INV-XXXXX or ORD-XXXXX.

        [OUTPUT FORMAT - MANDATORY]
        Current Plan: [The full list of steps]
        Current Step: [What you are doing now]
        Action: [TOOL_CALL: query_system("ID")] (If required)
        Final Answer: [Your message to the customer]
        """

    def track_tokens(self, response):
        """Accumulates total token count."""
        usage = response.usage_metadata
        self.total_tokens += usage.total_token_count
        return usage.prompt_token_count, usage.candidates_token_count

    def speak(self, user_message):
        """Executes Plan-and-Execute loop: Update plan -> Execute -> Reply"""
        current_input = f"{self.system_instruction}\n\n[USER MESSAGE]: {user_message}" if not self.chat.history else user_message
        
        t_p, t_c = 0, 0
        full_trajectory = []
        
        # Internal loop for Planning/Execution (Max 2 steps to save tokens)
        for i in range(2):
            response = self.chat.send_message(
                current_input, 
                generation_config=genai.types.GenerationConfig(
                    stop_sequences=["Observation:"],
                    temperature=0.0
                )
            )
            p, c = self.track_tokens(response)
            t_p += p; t_c += c
            
            out = response.text.strip()
            full_trajectory.append(out)
            
            # Update internal plan state
            plan_match = re.search(r'Current Plan:(.*?)Current Step:', out, re.DOTALL)
            if plan_match:
                self.plan = plan_match.group(1).strip()

            # Check for tool call
            match = re.search(r'\[TOOL_CALL: query_system\("(.*?)"\)\]', out)
            if match:
                s_id = match.group(1).strip()
                if s_id.isdigit(): s_id = f"INV-{s_id}"
                
                print(f"   ⚡ [P&E Executor] Executing tool: {s_id}")
                observation = ecommerce_system.query_system(s_id)
                
                full_trajectory.append(f"Observation: {observation}")
                current_input = f"Observation: {observation}\nUpdate your plan and provide the next 'Final Answer:'"
                continue 
            break

        # Extract only the Final Answer
        if "Final Answer:" in out:
            clean_reply = out.split("Final Answer:")[-1].strip()
        else:
            clean_reply = out.strip()
            
        return clean_reply, full_trajectory, t_p, t_c

# 3. Customer Proxy (Keep original)
class CustomerProxy:
    def __init__(self, fact_sheet):
        self.model = genai.GenerativeModel(MODEL_ID)
        self.fs = fact_sheet
        self.chat = self.model.start_chat()
        self.total_tokens = 0
        self.persona_prompt = f"""
        You are a customer who needs help. 
        Your specific goal: {self.fs['scenario_logic']['instruction']}
        
        YOUR DATA (Keep private until asked):
        - Invoice ID: {self.fs['ground_truth'].get('invoice_id')}
        - Order ID: {self.fs['ground_truth'].get('order_id')}
        - Email: {self.fs['ground_truth'].get('email')}

        BEHAVIOR RULES:
        1. FIRST MESSAGE: Briefly state your problem. DO NOT provide any ID or email.
        2. DO NOT reveal all data at once. Give ONLY the specific info the agent asks for.
        3. If the agent gives you a link or solves the problem, thank them and end the chat.
        """

    def track_tokens(self, response):
        self.total_tokens += response.usage_metadata.total_token_count

    def start_conversation(self):
        res = self.chat.send_message(f"{self.persona_prompt}\n\nPlease start the conversation.")
        self.track_tokens(res)
        return res.text

    def reply(self, agent_msg):
        res = self.chat.send_message(agent_msg)
        self.track_tokens(res)
        return res.text

# 4. Execution Function (Fix: Role Collapse / Plan Leakage)
def run_plan_execute_experiment(fact_sheet_path):
    with open(fact_sheet_path, 'r', encoding='utf-8') as f:
        fs = json.load(f)
    
    agent = PlanExecuteAgent()
    customer = CustomerProxy(fs)
    history = []

    print(f"\n[PLAN-AND-EXECUTE START] {fs['metadata']['conv_id']}")
    print("="*75)
    
    user_msg = customer.start_conversation()
    print(f"👤 CUSTOMER: {user_msg}")
    history.append({"role": "customer", "content": user_msg})
    
    for _ in range(6):
        # Support Agent Turn
        reply, trajectory, p_tok, c_tok = agent.speak(user_msg)
        
        # --- FIX: Role Collapse Bug ---
        # Aggressively remove any internal planning headers from the message sent to the customer
        clean_reply = re.sub(r'^(Current Plan|Current Step|Action|Thought|Observation):.*', '', reply, flags=re.MULTILINE | re.DOTALL).strip()
        # If the split didn't catch "Final Answer" label, clean it manually
        clean_reply = clean_reply.replace("Final Answer:", "").strip()
        
        print(f"🤖 AGENT: {clean_reply}")
        history.append({
            "role": "agent", "content": clean_reply, "trajectory": trajectory,
            "turn_tokens": {"prompt": p_tok, "response": c_tok}
        })
        
        if any(w in clean_reply.lower() for w in ["goodbye", "resolved", "have a nice day", "thank you"]):
            break
            
        # Customer Turn (Now receives ONLY the cleaned reply)
        user_msg = customer.reply(clean_reply)
        print(f"👤 CUSTOMER: {user_msg}")
        history.append({"role": "customer", "content": user_msg})

    # Save to JSON (Handles filename characters via conv_id sanitization if needed)
    safe_conv_id = re.sub(r'[^\w\s-]', '', fs['metadata']['conv_id']).strip().replace(' ', '_')
    os.makedirs("experiment_logs", exist_ok=True)
    log_path = f"experiment_logs/log_{safe_conv_id}_PlanExecute.json"
    
    log_data = {
        "metadata": fs['metadata'],
        "total_cost": {
            "agent": agent.total_tokens,
            "customer": customer.total_tokens,
            "grand_total": agent.total_tokens + customer.total_tokens
        },
        "history": history
    }
    with open(log_path, 'w', encoding='utf-8') as f:
        json.dump(log_data, f, indent=2, ensure_ascii=False)
    
    print("="*75)
    print(f"💰 Grand Total Tokens: {agent.total_tokens + customer.total_tokens}")
    print(f"📁 Log saved to: {log_path}")

# Run
run_plan_execute_experiment("fact_sheet_BITEXT_EASY_003.json")


[PLAN-AND-EXECUTE START] BITEXT_EASY_003
👤 CUSTOMER: Hi, I'm having trouble downloading an invoice from your website. Could you help me with that?
🤖 AGENT: Hi there! I'd be happy to help you download your invoice. Could you please provide me with your invoice or order ID? This will allow me to locate the correct document for you.
👤 CUSTOMER: My invoice ID is INV-37777.
🤖 AGENT: Thanks! Let me retrieve invoice INV-37777 for you. One moment please.
👤 CUSTOMER: Okay, thank you.
🤖 AGENT: Great! I have located invoice INV-37777. You can download it directly from this link: [fictional download link - example: https://example.com/invoice/INV-37777.pdf]. Please let me know if you have any trouble accessing the invoice.
👤 CUSTOMER: Perfect, that worked! Thank you so much for your help.
🤖 AGENT: You're very welcome! I'm glad I could help. If you have any other questions or need further assistance, please don't hesitate to ask. Have a great day!
👤 CUSTOMER: You too! Goodbye.
🤖 AGENT: Goodbye! Ha